In [ ]:
#user input (just run the remaining cells if no changes needed)

TOKEN = "BOT_TOKEN" #bot token
CHANNEL_ID = 1457548833333 #channel or thread id (right click channel/thread, then copy channel/thread id)
DEADLINE = "2026/01/05 15:00" #deadline in UTC
DROP_DUPLICATE = True #drop all multiple answers or not
INT_COL = ["Q1", "Q2", "Q3", "Q5"] #columns in integer, not float

In [4]:
pip install discord #if required

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#if': Expected package name at the start of dependency specifier
    #if
    ^


In [ ]:
#setting to scrap the messages
import discord
import pandas as pd


intents = discord.Intents.default() #create intents object to tell discord what to do
intents.message_content = True #Tell discord we are looking for message content

client = discord.Client(intents=intents) #create a client and use that intent

messages = [] 

@client.event #register an event to run once
async def on_ready(): #a coroutine that pause at await so other work can run while it’s waiting, instead of blocking the whole program
    #not run immediately
    print("Logged in as", client.user) #print the bot name

    channel = client.get_channel(CHANNEL_ID) #find the channel/thread
    if channel is None:
        print("Channel not found. Check CHANNEL_ID and that the bot is in the server.")
        await client.close() #close the discord
        return #end of the function

    print("Channel found:", channel.name) #print the channel name

    async for m in channel.history(): #run for the messages in channel
        messages.append({ #append the data in a list to the dictionary
            "author": str(m.author),
            "content": m.content,
            "created_at": m.created_at,
        })

    await client.close()


In [1]:
#run the bot to scrap
await client.start(TOKEN) #use await to start our event

NameError: name 'client' is not defined

In [8]:
#drop the first message to avoid treating the example as solution
df = pd.DataFrame(messages) 
df = df.iloc[:df.shape[0] - 1]

In [90]:
#split each line of messages
import re

rows = []

for _, row in df.iterrows():
    msg = row['content']
    for line in str(msg).splitlines():
        rows.append({
            'author': row['author'],
            'created_at': row['created_at'],
            'line': line,
        })

df_lines = pd.DataFrame(rows) 

In [ ]:
# pattern for filtering (require the message to have "[0-9]." before the answer)
pattern_filter = r'^\s*[0-9]+\s*\.\s*[0-9]+(?:\.[0-9]+)?'

mask = df_lines['line'].str.match(pattern_filter, na=False) #True False Series with NaN as False
df_answers = df_lines.loc[mask].copy() #filter out the df not in that pattern


In [ ]:
#start building the dataframe with q1/q2/... as column
df_answers['q'] = (
    df_answers['line']
    .str.extract(r'^\s*([0-9]+)\.', expand=False)
    .astype(int)
)

df_answers['answer'] = (
    df_answers['line']
    .str.extract(r'^\s*[0-9]+\.\s*([0-9]+(?:\.[0-9]+)?)\s*$', expand=False) # word, emoji after the answer will not be accepted
    .astype(float)
)

In [93]:
#finanlise the dataframe
wide = df_answers.pivot_table(
    index=['author', 'created_at'],
    columns='q',
    values='answer',
    aggfunc='first'
)

wide.columns = [f'Q{int(c)}' for c in wide.columns]  # 1→Q1, 2→Q2, ...
wide = wide.sort_values(["created_at"]).reset_index()

In [ ]:
#drop responses after deadline and change columns datatype (and duplicates)
wide = wide[wide.created_at < DEADLINE]
if DROP_DUPLICATE:
    keep_var = 0
else:
    keep_var = "last"
wide = wide.drop_duplicates(subset=["author"], keep = keep_var)
wide[INT_COL] = wide[INT_COL].astype("int64",)

In [ ]:
wide.to_csv("C:\Users\85251\Documents\2026cwc_comp1.xlsx", index = False)